# Merge and clean data layers

**Purpose:** Merge fecal metagenomic data, plasma metadata, diet variables, and metabolomics platforms into analysis-ready tables.

**Expected inputs**
- `../data/metadata/*.tsv or *.csv`
- `../data/metabolomics/*.csv`
- `../data/micov_filtered_feature-table.biom`

**Main outputs**
- `../data/plasma_metadata_matched_all_outcomes.tsv`
- `../data/plasma_metadata_matched_main_outcomes.tsv`
- `../data/*platform*.csv`

> Notes for reuse: data files are not included in this repository. Update paths in the cells below to match the local location of the approved, de-identified data release. Notebook outputs have been cleared for public sharing.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'output_files'
FIGURE_DIR = PROJECT_ROOT / 'figures'

for directory in [DATA_DIR, OUTPUT_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
from biom import load_table


## Merge data layers


In [ ]:
# Load fecal and plasma metadata

md_fecal = pd.read_csv('../data/metadata/adrc_metadata.tsv', sep = '\t', dtype=str)

md_plasma = pd.read_csv('../data/ADRC_NACCSubsetUDS_ADI_NO_NATIVE_AMER_for_654_Plasma_2024-10-24.csv', dtype={'Kit.Number': str}).set_index('Kit.Number')


In [ ]:
print(md_fecal.shape)
print(md_plasma.shape)


In [ ]:
# Load data with all platforms combined

plasma = pd.read_csv('../data/1-Plasma platforms combined_v3_2025-07-15.csv', dtype=str)


In [ ]:
# Next, for multiple samples from one participant, we will keep the plasma samples that were closest to fecal sample dates
plasma = plasma.dropna(subset='Fecal.plasma.days.diff')
plasma['Fecal.plasma.days.diff'] = plasma['Fecal.plasma.days.diff'].astype(int)

# Group by subject ID, and choose smallest number of days from fecal to plasma sample collection 
plasma = plasma.loc[plasma.groupby('NACC.ID')['Fecal.plasma.days.diff'].idxmin()]


In [ ]:

# Set index to match fecal and plasma samples by kit number
plasma = plasma.set_index('fecal_kit')
md_fecal = md_fecal.set_index('Kit.Number')


In [ ]:
# Drop any samples in fecal metadata that aren't present in plasma data and vice versa
md_fecal = md_fecal.loc[md_fecal.index.intersection(plasma.index)]
plasma = plasma.loc[plasma.index.intersection(md_fecal.index)]


In [ ]:
# Add sample name column to plasma data to match with feature table
plasma['sample_name'] = md_fecal['sample_name']


In [ ]:
# set index back to plasma kit number to match with plasma metadata, and add sample name to match feature table
plasma = plasma.set_index('Kit.Number Plasma', drop=False)
md_plasma = md_plasma.loc[md_plasma.index.intersection(plasma.index)]
md_plasma['sample_name'] = plasma['sample_name']


In [ ]:
# Set sample_name as index across both metadatas and plasma data
md_plasma = md_plasma.set_index('sample_name')
md_fecal = md_fecal.set_index('sample_name', drop=False)
plasma = plasma.set_index('sample_name', drop=False)


In [ ]:
# Import featuretable
ft = load_table('../data/micov_filtered_feature-table.biom')


In [ ]:
# Filter to samples present in plasma metadata and plasma data
ft = ft.filter(md_plasma.index, axis='sample')
plasma = plasma.loc[plasma.index.intersection(ft.ids(axis='sample'))]


In [ ]:
ft.to_dataframe().T.index.duplicated().sum()


In [ ]:
# Save matched data 
plasma.to_csv('../data/all_plasma_data_matched.csv')


## Clean metadata


In [ ]:
# Save new column as binary CU/CI
md_plasma['Diagnosis'] = md_plasma['NACCUDSD'].replace({1: 'Cognitively Unimpaired', 2: 'Cognitively Impaired', 3: 'Cognitively Impaired', 4: 'Cognitively Impaired'})


In [ ]:
# Replace missing values as np.nan
md_plasma['MOCA'] = md_plasma['NACCMOCA'].replace({-4.0: np.nan, 88.0: np.nan})
md_plasma['Antidepressants'] = md_plasma['NACCADEP'].replace({-4.0: np.nan})

md_plasma['UDSBENTD'] = np.where((md_plasma['UDSBENTD'] < 0) | (md_plasma['UDSBENTD'] > 17), np.nan, md_plasma['UDSBENTD'])
md_plasma['CRAFTDRE'] = np.where((md_plasma['CRAFTDRE'] < 0) | (md_plasma['CRAFTDRE'] > 25), np.nan, md_plasma['CRAFTDRE'])


In [ ]:
# Remove duplicated samples
md_plasma = md_plasma[~md_plasma.index.duplicated()]


In [ ]:
md_plasma = md_plasma[md_plasma['NACCAGE'] != 25]


In [ ]:
md_plasma['NACCBMI'].sort_values()


In [ ]:
md_plasma['NACCBMI'] = md_plasma['NACCBMI'].replace({-4.0: np.nan, 88.0: np.nan, 888.8: np.nan})


In [ ]:
md_plasma['HD-X pTau217'].value_counts()


In [ ]:
md_plasma['AMYLPET'] = md_plasma['AMYLPET'].replace({-4.0: np.nan, 8.0: np.nan})


In [ ]:
md_plasma['AMYLCSF'] = md_plasma['AMYLCSF'].replace({-4.0: np.nan, 8.0: np.nan})


In [ ]:
md_plasma['HIPPATR'] = md_plasma['HIPPATR'].replace({-4.0: np.nan, 8.0: np.nan})


In [ ]:
md_plasma['TAUPETAD'] = md_plasma['TAUPETAD'].replace({-4.0: np.nan, 8.0: np.nan})


In [ ]:
md_plasma['CSFTAU'] = md_plasma['CSFTAU'].replace({-4.0: np.nan, 8.0: np.nan})


In [ ]:
md_plasma['CSFTAU'].value_counts()


In [ ]:
# Rename columns
md_plasma = md_plasma.rename(columns={'DEP':'Depression', 'AMYLPET': 'Amyloid PET', 'AMYLCSF': 'Amyloid CSF', 'HIPPATR': 'Hippocampal Atrophy',
                                      'ANXIET': 'Anxiety', 'HD-X pTau181': 'pTau181', 'HD-X pTau217': 'pTau217', 'TAUPETAD': 'Tau PET','CSFTAU': 'Tau CSF',
                                     'HD-X NFL': 'NFL', 'HD-X GFAP': 'GFAP', 'Lumipulse ABeta40': 'Abeta40', 'Lumipulse ABeta42': 'Abeta42',
                                     'Most Likely Fasting[based on 4 platforms]': 'fasting'})


In [ ]:
# Merge ADI information
adi = pd.read_csv('../data/ADI_2022_PLASMA_NO_PHI_NO_NATIVE_AMER_2025-01-10_1439.csv')
adi = adi.rename(columns={'NACCID': 'NACC.ID'})
md_plasma = md_plasma.reset_index()
md_plasma = pd.merge(md_plasma, adi[['NACC.ID', 'ADI_NAT', 'ADI_STATE', 'site']], on='NACC.ID', how='left').set_index('sample_name')
md_plasma = md_plasma.rename(columns={'NACCID': 'NACC.ID', 'ADI_NAT_y': 'ADI_NAT', 'ADI_STATE_y': 'ADI_STATE'})


In [ ]:
# Merge diet information

diet = pd.read_csv('../data/metadata/diet_data_clean.csv', dtype={'sample_name': str}).set_index('sample_name')
diet = diet[~diet.index.duplicated()]
hei = diet.filter(like='HEI2015')
md_plasma = pd.merge(md_plasma, hei, how='left', right_index=True, left_index=True)


In [ ]:
md_plasma['fiber'] = diet['fiber']


In [ ]:
# Define outcomes 
outcomes = ['Diagnosis', 'fiber', 'Amyloid PET', 'CRAFTDRE','Amyloid CSF', 'Hippocampal Atrophy', 'Tau PET', 'Tau CSF', 'MOCA', 'UDSBENTD', 'Depression', 'Anxiety', 'Antidepressants', 'pTau181', 'pTau217', 'NFL', 'GFAP', 
            'Abeta40', 'Abeta42', 'ADI_NAT', 'ADI_STATE', 'site', 'NACCAGE', 'SEX', 'NACCBMI', 'fasting'] + list(md_plasma.filter(like='HEI2015').columns)


In [ ]:
md_plasma['Antidepressants'].value_counts()


In [ ]:
# Save metadatas

md_plasma.to_csv('../data/plasma_metadata_matched_all_outcomes.tsv', sep = '\t')
md_plasma_outcomes = md_plasma[outcomes]
md_plasma_outcomes.to_csv('../data/plasma_metadata_matched_main_outcomes.tsv', sep = '\t')


In [ ]:
ft = ft.filter(ids_to_keep=md_plasma.index, axis='sample')


In [ ]:
# save matched feature table
with open("../data/micov_filtered_feature-table_matched.biom", "w") as f:
    f.write(ft.to_json("Generated by script"))


## Clean Fecal Metabolomics/Foodomics


In [ ]:
foodomics = pd.read_csv('../data/Molecule Transition Results_ADRC_FecalsetOne.csv')


In [ ]:
set1 = pd.read_csv('../data/metabolomics/20240502_U19_ADRC_Set1_main_iimn_quant.csv')


In [ ]:
md = pd.read_csv('../data/metadata/adrc_metadata.tsv', sep = '\t', dtype=str)


In [ ]:
md = md.set_index('Specimen.Bar.Code')


In [ ]:
num = split.apply(lambda x: x[-1])


In [ ]:
foodomics.index = num


In [ ]:
foodomics = foodomics.loc[foodomics.index.intersection(md.index)]


In [ ]:
foodomics['sample_name'] = md['sample_name']


In [ ]:
foodomics = foodomics.set_index('sample_name')


In [ ]:
foodomics = foodomics.pivot_table(
    index='sample_name',
    columns='Molecule',
    values='Area_sub',
    # aggfunc='first'  # or 'mean' if duplicates exist
)


In [ ]:
foodomics.to_csv('../data/foodomics_adrc_fecal_set_one.csv')


## Annotate and clean plasma data layers


In [ ]:
# Load annotation file
annotations = pd.read_excel('../data/plasma_tables_by_platform/Variables and platforms.xlsx')


In [ ]:
# Keep metabolites present in plasma data

cols = {}

for platform in annotations['Platform'].unique():
    cols[platform] = []
    for metabolite in annotations[annotations['Platform'] == platform]['Variable']:
        if metabolite in plasma.columns:
            cols[platform].append(metabolite)
    else:
        print(f"{metabolite} from {platform} not in plasma dataframe")


In [ ]:
# Create separate feature tables per platform
dfs = {}

for platform in annotations['Platform'].unique():
    dfs[platform] = plasma[cols[platform]]


In [ ]:
# Count number of metabolites per platform

for platform in annotations['Platform'].unique():
    dfs[platform] = dfs[platform].loc[dfs[platform].index.intersection(md_plasma.index)]
    print(f'{platform} {dfs[platform].shape}')


In [ ]:
pd.read_csv('../data/metabolomics/20240502_U19_ADRC_Set1_main_iimn_quant.csv')


In [ ]:
# Save separated feature tables
for df in dfs.keys():
    dfs[df].to_csv(f'../data/{df}.csv')


In [ ]:
# Merge different Wishart datasets and save
wishart = pd.merge(dfs['Wishart Biocrates'], dfs['Wishart Water soluble vitamines'], left_index=True, right_index=True)
wishart = pd.merge(wishart, dfs['Wishart Fat soluble vitamines'], left_index=True, right_index=True)
wishart = pd.merge(wishart, dfs['Wishart Metals'], left_index=True, right_index=True)
wishart.to_csv('../data/wishart_combined.csv')


In [ ]:
# Combine and save Fiehn
fiehn = pd.merge(dfs['WCMC (FIEHN)'], dfs['WCMC Known'], left_index=True, right_index=True)
fiehn = pd.merge(fiehn, dfs['WCMC Unknown'], left_index=True, right_index=True)
fiehn.to_csv('../data/fiehn_combined.csv')


In [ ]:
# Combine and save UCSD
ucsd = pd.merge(dfs['UCSD Known'], dfs['UCSD unknown'], left_index=True, right_index=True)
ucsd.to_csv('../data/ucsd_combined.csv')


In [ ]:
# Combine and save tuulia
tuulia = pd.merge(dfs['Tuulia Known'], dfs['Tuulia unknown'], left_index=True, right_index=True)
tuulia.to_csv('../data/tuulia_combined.csv')


In [ ]:
# define new feature tables for upset plot
dfs = {
    'Nightingale': dfs['Nightingale'],
    'Baker': dfs['Baker'],
    'Metabolon': dfs['Metabolon'],
    'UCSD' : ucsd,
    'Metagenomics': ft.to_dataframe().T
}


In [ ]:
dfs_no_nans = {}

for df in dfs:
    print(df)
    dfs_no_nans[df] = dfs[df].dropna(how='all')
    print(dfs_no_nans[df].shape)


In [ ]:
# Step 1: Get all unique sample names across all datasets
all_samples = set().union(*[set(df.index) for df in dfs_no_nans.values()])

# Step 2: Create a binary matrix: rows = samples, cols = platforms
sample_matrix = pd.DataFrame(
    {platform: [sample in df.index for sample in all_samples] for platform, df in dfs_no_nans.items()},
    index=sorted(all_samples)
)


from upsetplot import from_indicators, UpSet
import matplotlib.pyplot as plt

# Convert binary matrix to UpSet format
data = from_indicators(sample_matrix, data=None)

# Plot
plt.figure(figsize=(12, 6))
UpSet(data, subset_size='count', show_counts=True, sort_by='cardinality').plot()
plt.title('Sample Overlap Across Platforms')
plt.tight_layout()
plt.savefig('../figures/sample_size_upset_plot.png', bbox_inches='tight')
plt.show()


In [ ]:
filtered_df = sample_matrix[sample_matrix.all(axis=1)]


In [ ]:
md_plasma = md_plasma.loc[filtered_df.index]


In [ ]:
# Save metadatas

md_plasma.to_csv('../data/plasma_metadata_matched_four_platforms.tsv', sep = '\t')
